In [1]:
print("Kernel is finally working")



Kernel is finally working


In [2]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

import shap
import joblib


ModuleNotFoundError: No module named 'seaborn'

In [ ]:
df = pd.read_csv("../data/anaemia.csv")
df.head()


,Gender,Hemoglobin,MCH,MCHC,MCV,Result
0,1,14.9,22.7,29.1,83.7,0
1,0,15.9,25.4,28.3,72.0,0
2,0,9.0,21.5,29.6,71.2,1
3,0,14.9,16.0,31.4,87.5,0
4,1,14.7,22.0,28.2,99.5,0


In [ ]:
X = df.drop("Result", axis=1)
y = df["Result"]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
svm = SVC(kernel="rbf", probability=True)
svm.fit(X_train_scaled, y_train)


,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,True
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [ ]:
y_pred = svm.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))


Accuracy: 0.9824561403508771


In [ ]:
joblib.dump(svm, "../model/svm_model.pkl")
joblib.dump(scaler, "../model/scaler.pkl")


['../model/scaler.pkl']

In [ ]:
explainer = shap.KernelExplainer(
    svm.predict_proba, X_train_scaled[:100]
)

shap_values = explainer.shap_values(X_test_scaled[:50])


100%|██████████| 50/50 [00:02<00:00, 20.57it/s]


In [ ]:
print("Type shap_values:", type(shap_values))
print("Length shap_values:", len(shap_values))

for i, sv in enumerate(shap_values):
    print(f"\nClass {i}")
    print("  type:", type(sv))
    print("  shape:", np.array(sv).shape)


Type shap_values: <class 'numpy.ndarray'>
Length shap_values: 50

Class 0
  type: <class 'numpy.ndarray'>
  shape: (5, 2)

Class 1
  type: <class 'numpy.ndarray'>
  shape: (5, 2)

Class 2
  type: <class 'numpy.ndarray'>
  shape: (5, 2)

Class 3
  type: <class 'numpy.ndarray'>
  shape: (5, 2)

Class 4
  type: <class 'numpy.ndarray'>
  shape: (5, 2)

Class 5
  type: <class 'numpy.ndarray'>
  shape: (5, 2)

Class 6
  type: <class 'numpy.ndarray'>
  shape: (5, 2)

Class 7
  type: <class 'numpy.ndarray'>
  shape: (5, 2)

Class 8
  type: <class 'numpy.ndarray'>
  shape: (5, 2)

Class 9
  type: <class 'numpy.ndarray'>
  shape: (5, 2)

Class 10
  type: <class 'numpy.ndarray'>
  shape: (5, 2)

Class 11
  type: <class 'numpy.ndarray'>
  shape: (5, 2)

Class 12
  type: <class 'numpy.ndarray'>
  shape: (5, 2)

Class 13
  type: <class 'numpy.ndarray'>
  shape: (5, 2)

Class 14
  type: <class 'numpy.ndarray'>
  shape: (5, 2)

Class 15
  type: <class 'numpy.ndarray'>
  shape: (5, 2)

Class 16
  type:

In [ ]:
print("X columns:", list(X.columns))
print("Number of features:", len(X.columns))


X columns: ['Gender', 'Hemoglobin', 'MCH', 'MCHC', 'MCV']
Number of features: 5


In [ ]:
# SHAP values for Anaemic class (class = 1)
shap_class = shap_values[1]   # shape: (n_samples, n_features)

print("SHAP class shape:", shap_class.shape)
print("Features:", len(X.columns))


SHAP class shape: (5, 2)
Features: 5


In [ ]:
# Mean absolute SHAP per feature
mean_shap = np.mean(np.abs(shap_class), axis=0)

print("Mean SHAP length:", len(mean_shap))


Mean SHAP length: 2


In [ ]:
# Select SHAP values for Anaemic class only
shap_anaemic = shap_class[:, 1]   # shape → (5,)


In [ ]:
shap_df = pd.DataFrame({
    "Feature": X.columns,
    "Mean_SHAP_Importance": np.abs(shap_anaemic)
}).sort_values(by="Mean_SHAP_Importance", ascending=False)

shap_df


,Feature,Mean_SHAP_Importance
1,Hemoglobin,0.313778
0,Gender,0.056026
4,MCV,0.010639
3,MCHC,0.001008
2,MCH,0.000385


In [ ]:
# Normalise features first
X_norm = (X - X.mean()) / X.std()

# Composite Anaemia Risk Index (explainable)
X["Anaemia_Risk_Index"] = (
    0.4 * X_norm["Hemoglobin"] +
    0.2 * X_norm["MCV"] +
    0.15 * X_norm["MCH"] +
    0.15 * X_norm["MCHC"] +
    0.1 * X_norm["Gender"]
)


In [ ]:

plt.figure(figsize=(10, 6))

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

In [ ]:
import numpy as np
import pandas as pd

# Z-score normalization (important)
X_norm = (X - X.mean()) / X.std()

# Explainable Composite Anaemia Risk Index (NOVEL FEATURE)
X["Anaemia_Risk_Index"] = (
    0.4 * X_norm["Hemoglobin"] +
    0.2 * X_norm["MCV"] +
    0.15 * X_norm["MCH"] +
    0.15 * X_norm["MCHC"] +
    0.1 * X_norm["Gender"]
)

# Verify
X.head()


,Gender,Hemoglobin,MCH,MCHC,MCV,Anaemia_Risk_Index
0,1,14.9,22.7,29.1,83.7,0.228295
1,0,15.9,25.4,28.3,72.0,0.004322
2,0,9.0,21.5,29.6,71.2,-1.418253
3,0,14.9,16.0,31.4,87.5,0.100141
4,1,14.7,22.0,28.2,99.5,0.392873


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Target
y = df["Result"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train SVM
svm = SVC(kernel="rbf", probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)


,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,True
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [ ]:
import shap

# Use small background set for KernelExplainer
background = X_train_scaled[:50]

explainer = shap.KernelExplainer(
    svm.predict_proba,
    background
)

# SHAP values for test samples
shap_values = explainer.shap_values(X_test_scaled[:50])


100%|██████████| 50/50 [00:05<00:00,  8.34it/s]


In [ ]:
# Anaemic class = class 1
shap_class = shap_values[1]  # shape: (features, classes)

print("SHAP class shape:", shap_class.shape)
print("Features:", X.columns.tolist())


SHAP class shape: (6, 2)
Features: ['Gender', 'Hemoglobin', 'MCH', 'MCHC', 'MCV', 'Anaemia_Risk_Index']


In [ ]:
# Select Anaemic class contribution (column index 1)
shap_anaemic = shap_class[:, 1]   # length = number of features

# Final SHAP contribution table
shap_df = pd.DataFrame({
    "Feature": X.columns,
    "Mean_SHAP_Importance": np.abs(shap_anaemic)
}).sort_values(by="Mean_SHAP_Importance", ascending=False)

shap_df


,Feature,Mean_SHAP_Importance
1,Hemoglobin,0.217675
0,Gender,0.111701
5,Anaemia_Risk_Index,0.082947
4,MCV,0.019424
3,MCHC,0.016997
2,MCH,0.011549


In [ ]:
# Pick one anaemic sample
idx = np.where(y_test == 1)[0][0]

x_original = X_test.iloc[idx]
x_scaled = scaler.transform([x_original])

# Original prediction
original_pred = svm.predict(x_scaled)[0]
original_prob = svm.predict_proba(x_scaled)[0]

original_pred, original_prob


c:\Users\anjuh\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


(np.int64(1), array([3.80604184e-11, 1.00000000e+00]))

In [ ]:
counterfactuals = []

for feature in X.columns:
    for delta in np.linspace(-2, 2, 41):  # small controlled change
        x_cf = x_original.copy()
        x_cf[feature] += delta

        x_cf_scaled = scaler.transform([x_cf])
        new_pred = svm.predict(x_cf_scaled)[0]

        if new_pred != original_pred:
            counterfactuals.append({
                "Feature_Changed": feature,
                "Change_Amount": round(delta, 2),
                "Original_Prediction": original_pred,
                "New_Prediction": new_pred
            })
            break


c:\Users\anjuh\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\anjuh\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\anjuh\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\anjuh\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\anjuh\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: U

In [ ]:
cf_df = pd.DataFrame(counterfactuals)
cf_df


,Feature_Changed,Change_Amount,Original_Prediction,New_Prediction
0,Anaemia_Risk_Index,1.8,1,0


In [ ]:
def clinical_rules_explanation(sample):
    explanations = []

    # Hemoglobin rules (WHO-based)
    if sample["Gender"] == 1:  # assuming 1 = Male, 0 = Female
        if sample["Hemoglobin"] < 13:
            explanations.append("Low hemoglobin detected for male patient")
    else:
        if sample["Hemoglobin"] < 12:
            explanations.append("Low hemoglobin detected for female patient")

    # MCV rules
    if sample["MCV"] < 80:
        explanations.append("Low MCV suggests microcytic anemia")
    elif sample["MCV"] > 100:
        explanations.append("High MCV suggests macrocytic anemia")

    # MCH rules
    if sample["MCH"] < 27:
        explanations.append("Low MCH indicates reduced hemoglobin per red blood cell")

    # MCHC rules
    if sample["MCHC"] < 32:
        explanations.append("Low MCHC indicates hypochromic red blood cells")

    # Fallback
    if len(explanations) == 0:
        explanations.append("Hematological parameters are within normal clinical ranges")

    return explanations


In [ ]:
# Use the same test sample
clinical_explanations = clinical_rules_explanation(x_original)

clinical_explanations


['Low hemoglobin detected for male patient',
 'Low MCV suggests microcytic anemia']

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,4))
sns.countplot(x='Result', data=df)
plt.title("Anaemic vs Non-Anaemic Distribution")
plt.xlabel("Result")
plt.ylabel("Count")

plt.tight_layout()
plt.savefig("../static/images/value_counts.png", dpi=300)
plt.close()


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='Gender', hue='Result', data=df)
plt.title("Gender-wise Anaemia Distribution")
plt.xlabel("Gender")
plt.ylabel("Count")

plt.tight_layout()
plt.savefig("../static/images/gender_distribution.png", dpi=300)
plt.close()


In [ ]:
plt.figure(figsize=(7,4))
sns.barplot(
    x="Mean_SHAP_Importance",
    y="Feature",
    data=shap_df,
    palette="viridis"
)

plt.title("SHAP Feature Importance")
plt.xlabel("Mean |SHAP Value|")
plt.ylabel("Feature")

plt.tight_layout()
plt.savefig("../static/images/shap_importance.png", dpi=300)
plt.close()


C:\Users\anjuh\AppData\Local\Temp\ipykernel_28584\2253286106.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(


In [ ]:
features = ["Hemoglobin", "MCV", "MCH", "MCHC"]

df[features].hist(figsize=(10,6), bins=20)
plt.suptitle("Distribution of Hematological Parameters")
plt.tight_layout()
plt.savefig("../static/images/feature_distribution.png", dpi=300)
plt.close()


In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df[features])
plt.title("Boxplot of Hematological Features")
plt.ylabel("Value")
plt.tight_layout()
plt.savefig("../static/images/boxplot_features.png", dpi=300)
plt.close()


In [ ]:
plt.figure(figsize=(7,5))
corr = df[features].corr()

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("../static/images/correlation_heatmap.png", dpi=300)
plt.close()


In [ ]:
plt.figure(figsize=(7,5))
corr = df[features].corr()

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("../static/images/correlation_heatmap.png", dpi=300)
plt.close()


In [ ]:
plt.figure(figsize=(6,4))
sns.barplot(
    x="Feature_Changed",
    y="Change_Amount",
    data=cf_df,
    palette="mako"
)

plt.title("Counterfactual Feature Change Required")
plt.ylabel("Change Amount")
plt.xlabel("Feature")

plt.tight_layout()
plt.savefig("../static/images/counterfactual_changes.png", dpi=300)
plt.close()


C:\Users\anjuh\AppData\Local\Temp\ipykernel_28584\3246981308.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(


In [ ]:
type(shap_values), len(shap_values)


(numpy.ndarray, 50)